In [11]:
#R H ARAVINDAN

In [22]:
import pandas as pd
import numpy as np
import re

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split

In [23]:
df = pd.read_csv("test.csv")
df.head()

,Class Index,Title,Description
0,3,Fears for T N pension after talks,Unions representing workers at Turner Newall...
1,4,The Race is On: Second Private Team Sets Launc...,"SPACE.com - TORONTO, Canada -- A second\team o..."
2,4,Ky. Company Wins Grant to Study Peptides (AP),AP - A company founded by a chemistry research...
3,4,Prediction Unit Helps Forecast Wildfires (AP),AP - It's barely dawn when Mike Fitzpatrick st...
4,4,Calif. Aims to Limit Farm-Related Smog (AP),AP - Southern California's smog-fighting agenc...


In [25]:
df["text"] = df["Title"].astype(str) + " " + df["Description"].astype(str)

X = df["text"]
y = df["Class Index"] - 1   # Labels: 0,1,2,3

print(df.shape)

(7600, 4)


In [26]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z ]', '', text)
    return text

X = X.apply(clean_text)

In [27]:
max_words = 10000
max_len = 100

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(X)

sequences = tokenizer.texts_to_sequences(X)

X_pad = pad_sequences(
    sequences,
    maxlen=max_len,
    padding="post",
    truncating="post"
)

print(X_pad.shape)

(7600, 100)


In [28]:
y = to_categorical(y, num_classes=4)

print(y.shape)

(7600, 4)


In [18]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z ]', '', text)
    return text

cleaned = clean_text(input_text)

In [31]:
X_train, X_test, y_train, y_test = train_test_split(X_pad,y,test_size=0.2,random_state=42)

In [33]:
model = Sequential([
    Embedding(input_dim=max_words,
              output_dim=64,
              ),

    SimpleRNN(64),

    Dense(32, activation="relu"),

    Dense(4, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)              │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ simple_rnn_1 (SimpleRNN)             │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [34]:
history = model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/5
152/152 ━━━━━━━━━━━━━━━━━━━━ 6s 24ms/step - accuracy: 0.2523 - loss: 1.3896 - val_accuracy: 0.2558 - val_loss: 1.3940
Epoch 2/5
152/152 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - accuracy: 0.2605 - loss: 1.3938 - val_accuracy: 0.2500 - val_loss: 1.3918
Epoch 3/5
152/152 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - accuracy: 0.2428 - loss: 1.3917 - val_accuracy: 0.2656 - val_loss: 1.3855
Epoch 4/5
152/152 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - accuracy: 0.2572 - loss: 1.3911 - val_accuracy: 0.2344 - val_loss: 1.3994
Epoch 5/5
152/152 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.2418 - loss: 1.4000 - val_accuracy: 0.2656 - val_loss: 1.3943


In [35]:
loss, accuracy = model.evaluate(X_test, y_test)

print("Accuracy:", accuracy)

48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.2349 - loss: 1.3966
Accuracy: 0.23486842215061188


In [36]:
label_map = {
    0: "World",
    1: "Sports",
    2: "Business",
    3: "Sci/Tech"
}

news = """
Apple launches a new AI powered iPhone with advanced features.
"""

cleaned = clean_text(news)

sequence = tokenizer.texts_to_sequences([cleaned])

padded = pad_sequences(
    sequence,
    maxlen=max_len,
    padding="post"
)

prediction = model.predict(padded)

pred_class = np.argmax(prediction)

print("Predicted Class:", label_map[pred_class])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 372ms/step
Predicted Class: Business
